# 08. Симуляция обратной связи и A/B против популярности

Офлайн-метрики (E01–E09) отвечают на вопрос «найдёт ли модель спрятанное аниме **за один показ**». В продукте лента показывается много раз, и между показами всё меняется:

- **обратная связь пользователя:** досмотренное из ленты попадает в историю – следующая лента строится уже по ней;
- **обратная связь системы:** досмотры увеличивают популярность – популярное показывают ещё чаще («богатые богатеют»).

Гипотеза: система (iALS → CatBoost → правила) за 10 показов даёт больше досмотров, чем популярная лента с теми же правилами, и показывает заметно больше разных аниме.

Код: `backend/training/src/training/simulation/feedback.py`. Лента строится **тем же классом, что в сервисе** (`serving.recommender.Recommender`), но с моделями, которые не видели test-пользователей: iALS переобучается на train, CatBoost из `models/ranker_v0` и так обучен на train-кандидатах, популярность – по train.

## Модель пользователя

Её нельзя измерить без логов показов, поэтому её задаём заранее и не подбираем:

| Допущение | Значение |
| --- | --- |
| Пользователи | 173 test-пользователя, приходят со своей input-историей |
| Раунды | 10 показов ленты по 10 аниме |
| Внимание | позицию `i` пользователь рассматривает с вероятностью `1 / log2(i + 1)`: 1,00; 0,63; 0,50; … 0,29 |
| Досмотр | рассмотренное досматривается, только если оно из его спрятанных ответов |
| Надоедание | аниме, показанное 2 раза без досмотра, больше не показываем |
| Популярность | досмотры раунда прибавляются к числу зрителей |
| Случайность | одинаковые броски в обоих вариантах: разница – от лент, а не от удачи |

Главное ограничение: «досмотрит только спрятанное» – та же метка, на которую обучен CatBoost. Симуляция показывает, как эффект модели складывается за несколько показов, но не меряет, понравится ли лента живым людям.

In [ ]:
import matplotlib.pyplot as plt

from core import logs
from training.simulation import feedback

logs.enable("INFO")
results = feedback.simulate()  # ~4 минуты: 173 пользователя × 10 раундов × 2 варианта

## Итог по вариантам

- `досмотров на пользователя` – за 10 раундов, с 95% интервалом (`low`, `high`);
- `доля с досмотром` – у скольких пользователей был хотя бы один досмотр;
- `доля найденных ответов` – какая часть спрятанного досмотрена (как recall, но с вниманием и повторными показами);
- `покрытие каталога` – какая доля каталога хоть раз попала в показы;
- `джини показов` – насколько показы сосредоточены на немногих аниме (0 – поровну, 1 – всё одному);
- `популярность показанного` – средний `log(1 + число зрителей)` показанных аниме.

In [ ]:
summary = feedback.arm_summary(results)
summary.round(3)

## Накопленные досмотры и покрытие по раундам

Два графика рядом, а не одна ось на двоих: у досмотров и покрытия разные единицы.

In [ ]:
colors = {"популярность": "#eb6834", "система": "#2a78d6"}
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for arm, result in results.items():
    rounds = result.per_round.index
    left.plot(rounds, result.per_round["watched"], color=colors[arm], linewidth=2, marker="o", markersize=5, label=arm)
    right.plot(rounds, result.per_round["coverage"], color=colors[arm], linewidth=2, marker="o", markersize=5, label=arm)
    # подпись последней точки вместо числа у каждой
    left.annotate(f"{result.per_round['watched'].iloc[-1]:.1f}", (rounds[-1], result.per_round["watched"].iloc[-1]),
                  xytext=(6, 0), textcoords="offset points", va="center")
    right.annotate(f"{result.per_round['coverage'].iloc[-1]:.0%}", (rounds[-1], result.per_round["coverage"].iloc[-1]),
                   xytext=(6, 0), textcoords="offset points", va="center")
left.set_title("Досмотров на пользователя, накопленно")
right.set_title("Доля каталога, показанная хотя бы раз")
right.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
for axis in (left, right):
    axis.set_xlabel("раунд (показ ленты)")
    axis.set_xticks(range(1, feedback.ROUNDS + 1))
    axis.grid(axis="y", color="#e2e0db", linewidth=0.8)
    axis.spines[["top", "right"]].set_visible(False)
    axis.legend(frameon=False)
plt.tight_layout()
plt.show()

## Симулированный A/B

В настоящем A/B-тесте каждый пользователь видит только один вариант: пользователей делят случайно пополам, и разница средних считается между **разными** людьми. В симуляции можно больше – показать обоим вариантам одних и тех же людей (парное сравнение). Сравниваем, насколько шире интервал у честного A/B.

In [ ]:
ab = feedback.ab_test(results)
ab.round(3)

Вывод (числа – из прогона `uv run python -m training.simulation.feedback`, `seed=42`):

- **Система даёт почти вдвое больше досмотров:** 11,8 [10,4; 13,3] на пользователя за 10 показов против 6,2 [5,4; 7,2] у популярной ленты. Хотя бы один досмотр – у 97% пользователей против 88%; найдено 35% спрятанного против 18%.
- **Разрыв появляется сразу и растёт.** После первого показа 2,6 против 1,1 досмотра, дальше обе кривые замедляются: найденное уходит в историю, надоевшее перестаёт показываться, спрятанного становится меньше.
- **Популярная лента застревает в узкой части каталога:** за 10 показов она показала 6% каталога, система – 31%. Показы у популярности сосредоточеннее (Джини 0,66 против 0,56), и показанное у неё в среднем популярнее (`log(1 + зрителей)` 6,15 против 5,22).
- **A/B:** на случайных половинах (87 и 86 человек) разница +6,3 [3,9; 8,5] досмотра, на тех же людях +5,6 [4,8; 6,5]. Оба интервала далеко от нуля, но у честного A/B он примерно в 2,5 раза шире: общая «сложность» пользователя не вычитается. При эффекте поменьше 173 человек в настоящем тесте могло бы не хватить – мощность теста надо считать до запуска.
- **Петля популярности здесь слабая.** 173 пользователя прибавляют к ~1650 train-зрителям мало, поэтому «богатые богатеют» за 10 раундов почти не видно; в продукте, где популярность пересчитывается по всем показам сайта, эффект сильнее.
- **Чего симуляция не показывает.** Пользователь досматривает только спрятанное – на это же обучен CatBoost, поэтому результат частично повторяет офлайн-метрику. Как люди реагируют на незнакомые им аниме, покажет только настоящий A/B с логами показов.